## Mini-demo of Graph RAG applied to healthcare

### Setup

In [1]:
from pathlib import Path
import yaml
from assets.graph_rag_demo.graph.engine import GraphRAG
from assets.graph_rag_demo.graph.adapters_sets import add_sets_and_policies

# Define the Graph RAG object -- all relevant data and code are in assets/graph_rag_deo
eng = GraphRAG(Path("assets/graph_rag_demo/node_docs.yaml"))
# Load policies
with open("assets/graph_rag_demo/data/policy_sets.yaml", "r") as f:
    policy_sets = yaml.safe_load(f)
add_sets_and_policies(eng, policy_sets)



### Demo - show results from several test cases

In [2]:
tests = [
 "Is arthrocentesis (20610) covered for M75.01 and M75.41?",
 "Is joint injection (20552) covered for M75.41?",
 "Bilateral arthrocentesis (20610) for M75.01 — any modifier?",
 "Is arthrocentesis (20610) covered for M54.2?",
 "Is 20610 covered for M75.11?",
 "Is 20552 covered for M75.11?",
]
for q in tests:
    print("Q:", q)
    print(eng.answer(q), "\n"+"-"*80+"\n")

Q: Is arthrocentesis (20610) covered for M75.01 and M75.41?
A: Yes — covered when medically necessary for the indicated diagnosis.

Per-diagnosis:
  M750*: ✓ qualifies under cms-arthro via DxSet[icd:frozen_shoulder]
  M7500: ✗ no matching policy set
  M7501: ✓ qualifies under cms-arthro via DxSet[icd:frozen_shoulder]
  M7502: ✗ no matching policy set
  M7541: ✗ matches DxSet[icd:shoulder_impingement] but not with this procedure

Why (reason path):
  Diagnosis:M7541 --IS_IN→ DxSet:icd:shoulder_impingement
  Diagnosis:M7501 --IS_IN→ DxSet:icd:frozen_shoulder
  Diagnosis:M7501 --IS_A→ Diagnosis:M750*
  Diagnosis:M750* --IS_IN→ DxSet:icd:frozen_shoulder
  Procedure:20610 --IS_IN→ ProcSet:cpt:arthro_block
  Coverage:cms-injection --USES_DXSET→ DxSet:icd:shoulder_impingement
  Coverage:cms-arthro --USES_PROCSET→ ProcSet:cpt:arthro_block
  Coverage:cms-arthro --USES_DXSET→ DxSet:icd:frozen_shoulder
  Diagnosis:M7500 --IS_A→ Diagnosis:M750*
  Diagnosis:M7502 --IS_A→ Diagnosis:M750*

Matched se

## Raw Data Collection

#### Generate sample procedure codes

In [ ]:
import csv
rows = []
rows.append(("PROC-123", "Arthrocentesis (demo)"))
rows.append(("PROC-456", "Nerve conduction study (demo)"))
for code in [20600,20605,20610,20550,20551,20552,20553]:
    rows.append((str(code), f"Demo procedure {code}"))
# add a synthetic block 20700–20749 to hit ~100 rows
for code in range(20700, 20750):
    rows.append((str(code), f"Demo procedure {code}"))

with open("assets/graph_rag_demo/data/procedures.csv", "w", newline="") as f:
    w = csv.writer(f, delimiter="|")
    w.writerow(["code","name"])
    w.writerows(rows)
print("Wrote", len(rows), "procedures")

#### Download ICD code descriptions

In [ ]:
import re
from pathlib import Path
import pandas as pd
import requests

# 1) Download the CMS ZIP (adjust URL to the current year's "Code Descriptions in Tabular Order")
url = "https://ftp.cdc.gov/pub/health_statistics/nchs/publications/ICD10CM/2025-Update/Code-desciptions-April-2025.zip"
zbytes = requests.get(url, timeout=60).content
# Then unzip


In [ ]:

IN_PATH  = Path("/Users/douglasdaly/Downloads/Code-desciptions-April-2025/icd10cm-codes-April-2025.txt")
OUT_PATH = Path("/Users/douglasdaly/Documents/GitHub/Generative-AI/assets/graph_rag_demo/data/icd10.csv")

# Your exact requirement for the code: [A-Z][0-9]+
# Separator is 2+ spaces, second column is the remainder.
LINE_RE = re.compile(r'^\s*([^"]*?)\s*$')  # first pass: strip quotes safely
SPLIT_RE = re.compile(r'^\s*([A-Z0-9\.]+)\s{1,}(.*?)\s*$')

def parse_icd_lines(in_path: Path):
    rows = []
    skipped = 0
    with in_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            # Remove all double quotes and surrounding whitespace/newlines
            line = line.replace('"', '')
            if 1:
                m = SPLIT_RE.match(line)
                if not m:
                    print(line)
                    skipped += 1
                    continue
                code, desc = m.group(1), m.group(2)
                rows.append((code, desc))
            else:
                rows.append(line)
    return rows, skipped

def main():
    rows, skipped = parse_icd_lines(IN_PATH)
    if not rows:
        raise SystemExit(f"No rows parsed from {IN_PATH}. Check the input format.")
    df = pd.DataFrame(rows, columns=["code", "name"])
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_PATH, sep="|", index=False, encoding="utf-8")
    print(f"Wrote {len(df):,} rows to {OUT_PATH}")
    if skipped:
        print(f"Skipped {skipped:,} lines that didn’t match the pattern (ok if the file has headers/footers).")

if __name__ == "__main__":
    main()


Wrote 74,260 rows to /Users/douglasdaly/Documents/GitHub/Generative-AI/assets/graph_rag_demo/data/icd10.csv


Recommended demo queries & expected results

Below are concise “truth tables” for the scaled mini-corpus we set up:

Policies/sets assumed

cms-arthro: ProcSets = cpt:arthro_block (20600–20611); DxSets = icd:frozen_shoulder (M75.0x), icd:rotator_cuff (M75.1x)

cms-injection: ProcSets = cpt:joint_injection (20552,20553); DxSets = icd:shoulder_impingement (M75.4x), icd:rotator_cuff (M75.1x)

(Optional) cms-tendon: ProcSets = cpt:tendon_injection (20550,20551); DxSets = icd:tendinitis (M75.2x), icd:rotator_cuff (M75.1x)

A) Single PX, two DX (only one qualifies)

Q: Is arthrocentesis (20610) covered for M75.01 and M75.41?
Expected: Yes — policy cms-arthro via M75.01 ∈ icd:frozen_shoulder.
Per-DX:

M75.01 → ✓ qualifies under cms-arthro

M75.41 → ✗ matches DxSet (shoulder_impingement) but not with this procedure

B) Injection that pairs with impingement

Q: Is joint injection (20552) covered for M75.41?
Expected: Yes — policy cms-injection (20552 ∈ cpt:joint_injection, M75.41 ∈ icd:shoulder_impingement).

C) Modifiers

Q: Bilateral arthrocentesis (20610) for M75.01 — any modifier?
Expected: Yes (cms-arthro). Note: append -50 (if bilateral).

D) DX that doesn’t qualify for 20610

Q: Is 20610 covered for M54.2 (cervicalgia)?
Expected: No — no policy where 20610 ProcSet intersects a DxSet containing M54.2x.

E) Same DX, different PX → different policies

Q1: Is 20610 covered for M75.11 (rotator cuff)?
Expected: Yes — cms-arthro (DX in icd:rotator_cuff, PX in cpt:arthro_block).
Q2: Is 20552 covered for M75.11?
Expected: Yes — cms-injection (DX in icd:rotator_cuff, PX in cpt:joint_injection).

F) Family vs specific

Q: Is 20610 covered for M75.0x vs M75.01?
Expected: Yes to both — roll-up and specific both hit icd:frozen_shoulder.

G) Tendon injections (optional policy)

Q: Is 20550 covered for M75.21 (tendinitis)?
Expected: Yes — cms-tendon (if you included the tendon policy).
Q: Is 20550 covered for M75.41?
Expected: No — PX hits tendon_injection; DX hits shoulder_impingement; no shared policy.